# Validando valores default

Vimos que o Pydantic, por padrão, não valida valores default.

Ou seja, a responsabilidade é nossa, que escrevemos o modelo, de garantir que os defaults respeitem o tipo e as validações do campo.

Mas podemos indicar, no nível do modelo (ou do campo, que veremos depois), que os defaults também devem ser validados.

Isso costuma ser útil quando a validação do default provoca alguma transformação nos dados (ou preenche outro campo a partir deles), e
queremos que as mesmas transformações aconteçam venha o valor dos dados ou do default.

Não validar defaults pode nos levar, por engano, a definir campos assim:

In [1]:
from pydantic import BaseModel, ConfigDict, ValidationError

class Model(BaseModel):
    field_1: int = None
    field_2: str = 100

`field_1` não é anulável, mas o default é `None` — e nenhuma exceção ocorre quando o default é aplicado. O mesmo com `field_2`:

In [2]:
print(f"Model(): {Model()!r}")

Model(): Model(field_1=None, field_2=100)


Porém, não podemos mandar `None` para `field_1` nos dados, nem um inteiro para `field_2`:

> 📝 **Nota de revisão:** no original, este parágrafo estava numa célula de **código** (o que gera `SyntaxError` ao executar o notebook);
> aqui virou markdown.

In [3]:
try:
    Model(field_1=None, field_2=100)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Model
field_1
  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/int_type
field_2
  Input should be a valid string [type=string_type, input_value=100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


Podemos habilitar a validação de defaults no nível do modelo (vale para todos os campos, a menos que seja desligada em campos
específicos) com `model_config`.

In [4]:
class Model(BaseModel):
    model_config = ConfigDict(validate_default=True)

    field_1: int = None
    field_2: str = 100

In [5]:
try:
    Model()
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Model
field_1
  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/int_type
field_2
  Input should be a valid string [type=string_type, input_value=100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
